In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def normalize_g2(g2: torch.Tensor, min_val=1.0, max_val=1.15) -> torch.Tensor:
    g2_masked_diag = g2.clone()
    g2_masked_diag.fill_diagonal_(g2_masked_diag.mean())
    g2_min, g2_max = g2_masked_diag.min(), g2_masked_diag.max()
    normalized_g2 = (g2 - g2_min) / (g2_max - g2_min + 1e-8) * (max_val - min_val) + min_val
    normalized_g2.fill_diagonal_(1.2)
    return normalized_g2

def as_float_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().squeeze().numpy()
    return np.asarray(x, dtype=np.float32)


def normalize_to_report_range(x, low=1.0, high=1.2):
    tensor = torch.as_tensor(as_float_array(x), dtype=torch.float32)
    return as_float_array(normalize_g2(tensor, min_val=low, max_val=high))


def diagonal_mean_matrix(x):
    x = as_float_array(x)
    n = x.shape[0]
    out = np.empty_like(x)
    for offset in range(-n + 1, n):
        mean_value = float(np.diagonal(x, offset=offset).mean())
        if offset >= 0:
            idx = np.arange(n - offset)
            out[idx, idx + offset] = mean_value
        else:
            idx = np.arange(n + offset)
            out[idx - offset, idx] = mean_value
    return out


def nearest_index(times, value):
    return int(np.argmin(np.abs(times - value)))


def load_experiment_g2(npz_path, channel=0, t_max_s=2500):
    npz_path = str(npz_path)
    if npz_path.endswith(".npz"):
        with np.load(npz_path) as data:
            g2 = data["g12"]
            t_max_s = min(t_max_s, g2.shape[0] - 1, g2.shape[1] - 1)
            g2 = g2[: t_max_s + 1, : t_max_s + 1, channel]
    elif npz_path.endswith(".npy"):
        g2 = np.load(npz_path)
        if g2.ndim == 3:
            g2 = g2[:, :, channel]
        t_max_s = min(t_max_s, g2.shape[0] - 1, g2.shape[1] - 1)
        g2 = g2[: t_max_s + 1, : t_max_s + 1]
    times = np.linspace(0, t_max_s, g2.shape[0])
    return normalize_to_report_range(g2), times


def load_simulation_g2(pt_path, t_max_s=2500):
    g2 = torch.load(pt_path, map_location="cpu", weights_only=True).squeeze(0)
    g2 = as_float_array(g2)
    times = np.linspace(0, t_max_s, g2.shape[-1])
    return g2, times

In [ ]:
# Experimental examples: 2NSN10_16_2019_dose0, first 2500 s.
# The T193C file metadata is 466.15 K; the panel label below follows the requested 468 K wording.
experiment_specs = [
    {"label": "299 K", "path": PROJECT_ROOT / "exp_data" / "2NSN10_16_2019_dose0_T25C.npy"},
    {"label": "371 K", "path": PROJECT_ROOT / "exp_data" / "2NSN10_16_2019_dose0_T98C.npy"},
    {"label": "421 K", "path": PROJECT_ROOT / "exp_data" / "2NSN10_16_2019_dose0_T148C.npy"},
    {"label": "471 K", "path": PROJECT_ROOT / "exp_data" / "2NSN10_16_2019_dose0_T198C.npy"},
]

experiment_panels = []
for spec in experiment_specs:
    g2, times = load_experiment_g2(spec["path"], channel=0, t_max_s=2500)
    experiment_panels.append({**spec, "g2": g2, "times": times, "diag_mean": diagonal_mean_matrix(g2)})

[(panel["label"], panel["g2"].shape, float(panel["g2"].min()), float(panel["g2"].max())) for panel in experiment_panels]

In [ ]:
# Plain-text non-equilibrium measures for the experimental panels.
for panel in experiment_panels:
    residual = panel["g2"] - panel["diag_mean"]
    denominator = panel["g2"] - panel["g2"].mean()
    non_eq = np.linalg.norm(residual) / np.linalg.norm(denominator)
    print(f"030BM_L_dose3 {panel['label']}: non-equilibrium measure = {non_eq:.6f}")

In [ ]:
# Global plotting settings. Keep panel-specific style choices in the plotting cells below.
plt.rcParams["font.family"] = "arial"
plt.rcParams["font.size"] = 28
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'arial'
plt.rcParams['mathtext.it'] = 'arial:italic'
plt.rcParams['mathtext.bf'] = 'arial:bold'
plt.rcParams['mathtext.cal'] = 'cmsy10'

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(30, 6), sharex=True, sharey=True)
image = None
for ax, panel in zip(axes, experiment_panels):
    image = ax.imshow(
        panel["g2"],
        origin="lower",
        extent=[panel["times"][0], panel["times"][-1], panel["times"][0], panel["times"][-1]],
        vmin=1.0,
        vmax=1.2,
        cmap="viridis",
        # interpolation="nearest",
        # rasterized=True,
    )
    # ax.set_title(panel["label"])
    ax.set_xlabel(r"Time $t_2$ (s)")
    ax.set_xticks([0, 128, 256], [0, 1250, 2500])
    ax.set_yticks([0, 128, 256], [0, 1250, 2500])
axes[0].set_ylabel(r"Time $t_1$ (s)")
colorbar = fig.colorbar(image, ax=axes, location="right", pad=0.035)
colorbar.set_label(r"$g_2(t_1,t_2)$")
colorbar.set_ticks([1.0, 1.1, 1.2])
fig.savefig(OUTPUT_DIR / "2NSN10_16_2019_dose0_aligned.pdf", bbox_inches="tight")
plt.show()

In [ ]:
slice_times_s = [0, 500, 1000, 1500, 2000]
slice_colors = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e"]

fig, axes = plt.subplots(1, 4, figsize=(26, 6), sharex=True, sharey=True)
for ax, panel in zip(axes, experiment_panels):
    for t1, color in zip(slice_times_s, slice_colors):
        i = nearest_index(panel["times"], t1 * 256 / 2500)
        tau = panel["times"][i:] - panel["times"][i]
        curve = panel["g2"][i, i:]
        ax.plot(tau, curve, lw=2, color=color, label=rf"$t_1 =${t1:g} s")
    # ax.text(0.04, 0.94, panel["label"], transform=ax.transAxes, ha="left", va="top")
    ax.set_xlabel(r"Lag time $\tau$ (s)")
    ax.set_xlim(-10, 256)
    ax.set_xticks([0, 128, 256], [0, 1250, 2500])
    ax.set_ylim(0.98, 1.22)
axes[0].set_ylabel(r"$f(\tau;t_1)=g_2(t_1,t_1+\tau)$")
axes[-1].legend(frameon=True, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.savefig(OUTPUT_DIR / "2NSN10_16_2019_dose0_correlation_slices.pdf", bbox_inches="tight")
plt.show()

In [ ]:
experiment_residuals = [panel["g2"] - panel["diag_mean"] for panel in experiment_panels]
residual_limit = max(float(np.nanmax(np.abs(residual))) for residual in experiment_residuals)
residual_limit = min(max(residual_limit, 0.02), 0.2)

fig, axes = plt.subplots(1, 4, figsize=(30, 6), sharex=True, sharey=True)
image = None
norm = TwoSlopeNorm(vmin=-residual_limit, vcenter=0.0, vmax=residual_limit)
for ax, panel, residual in zip(axes, experiment_panels, experiment_residuals):
    image = ax.imshow(
        residual,
        origin="lower",
        extent=[panel["times"][0], panel["times"][-1], panel["times"][0], panel["times"][-1]],
        cmap="PiYG",
        norm=norm,
        interpolation="nearest",
        rasterized=True,
    )
    ax.set_xlabel(r"Time $t_2$ (s)")
    ax.set_xticks([0, 128, 256], [0, 1250, 2500])
    ax.set_yticks([0, 128, 256], [0, 1250, 2500])
axes[0].set_ylabel(r"Time $t_1$ (s)")
colorbar = fig.colorbar(image, ax=axes, location="right", pad=0.035)
# colorbar.set_ticks([-0.5, 0.0, 0.5])
colorbar.set_label(r"$g_2(t_1,t_2)-g_2^{\mathrm{TTI}}(t_2-t_1)$")
fig.savefig(OUTPUT_DIR / "2NSN10_16_2019_dose0_g2_minus_diag_mean.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def normalize_g2(g2: torch.Tensor, min_val=1.0, max_val=1.15) -> torch.Tensor:
    g2_min, g2_max = g2.min(), g2.max()
    normalized_g2 = (g2 - g2_min) / (g2_max - g2_min + 1e-8) * (max_val - min_val) + min_val
    return normalized_g2

In [ ]:
simulation_manifest = PROJECT_ROOT / "dataset" / "simulation_2" / "manifest_with_non_equ.csv"
simulation_df = pd.read_csv(simulation_manifest)

rng = np.random.default_rng(20050628)
ranked = simulation_df.sort_values("nonequilibrium_measure_raw", ascending=True).reset_index().rename(columns={"index": "manifest_row"})
noneq_regions = [
    (0.0, 0.2),
    (0.2, 0.4),
    (0.4, 0.7),
    (0.7, 1.0),
]
selected_rows = []
for region_index, (region_min, region_max) in enumerate(noneq_regions, start=1):
    values = ranked["nonequilibrium_measure_raw"]
    if region_index == len(noneq_regions):
        in_region = values.between(region_min, region_max, inclusive="both")
    else:
        in_region = (values >= region_min) & (values < region_max)

    candidates = ranked.loc[in_region]
    if candidates.empty:
        raise ValueError(f"No simulation rows found with raw non-equilibrium measure in [{region_min}, {region_max}]")

    rank_position = int(rng.choice(candidates.index.to_numpy()))
    row = ranked.loc[rank_position].copy()
    row["noneq_region"] = region_index
    row["region_min"] = region_min
    row["region_max"] = region_max
    row["region_label"] = f"R{region_index}: [{region_min:g}, {region_max:g}{']' if region_index == len(noneq_regions) else ')'}"
    row["rank_position"] = rank_position
    selected_rows.append(row)

selected_simulations = pd.DataFrame(selected_rows)
selected_simulations = selected_simulations[[
    "noneq_region", "region_label", "region_min", "region_max", "rank_position", "manifest_row", "id", "gamma", "D", "GB_conc", "T", "nonequilibrium_measure_raw", "nonequilibrium_measure", "path"
]]
selected_simulations.to_csv(OUTPUT_DIR / "simulation_noneq_region_selection.csv", index=False)
selected_simulations

In [ ]:
simulation_panels = []
for _, row in selected_simulations.iterrows():
    g2, times = load_simulation_g2(PROJECT_ROOT / row["path"], t_max_s=2500)
    g2_panel = normalize_g2(g2, min_val=1.0, max_val=1.2)
    simulation_panels.append({
        "label": f"R{int(row['noneq_region'])}",
        "g2": g2_panel,
        "times": times,
        "diag_mean": diagonal_mean_matrix(g2_panel),
        "row": row,
    })

[(panel["label"], panel["g2"].shape, float(panel["g2"].min()), float(panel["g2"].max())) for panel in simulation_panels]

In [ ]:
# Plain-text non-equilibrium measures for the selected simulation panels.
# The recomputed value should match nonequilibrium_measure_raw; selection uses raw fixed-width regions.
for panel in simulation_panels:
    row = panel["row"]
    residual = panel["g2"] - panel["diag_mean"]
    denominator = panel["g2"] - panel["g2"].mean()
    non_eq = np.linalg.norm(residual) / np.linalg.norm(denominator)
    print(
        f"{panel['label']} id {int(row['id'])}: "
        f"non-equilibrium measure = {non_eq:.6f} "
        f"(manifest raw = {row['nonequilibrium_measure_raw']:.6f}, "
        f"region = {row['region_label']})"
    )

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(30, 6), sharex=True, sharey=True)
image = None
for ax, panel in zip(axes, simulation_panels):
    image = ax.imshow(
        panel["g2"],
        origin="lower",
        extent=[panel["times"][0], panel["times"][-1], panel["times"][0], panel["times"][-1]],
        vmin=1.0,
        vmax=1.2,
        cmap="viridis",
    )
    # ax.text(0.04, 0.94, panel["label"], transform=ax.transAxes, ha="left", va="top", color="white")
    ax.set_xlabel(r"Time $t_2$ (s)")
    ax.set_xticks([0, 1250, 2500])
    ax.set_yticks([0, 1250, 2500])
axes[0].set_ylabel(r"Time $t_1$ (s)")
colorbar = fig.colorbar(image, ax=axes, location="right", pad=0.035)
colorbar.set_label(r"$g_2(t_1,t_2)$")
colorbar.set_ticks([1.0, 1.1, 1.2])
fig.savefig(OUTPUT_DIR / "simulation_noneq_regions_g2_aligned.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(26, 6), sharex=True, sharey=True)
for ax, panel in zip(axes, simulation_panels):
    for t1, color in zip(slice_times_s, slice_colors):
        i = nearest_index(panel["times"], t1)
        tau = panel["times"][i:] - panel["times"][i]
        curve = panel["g2"][i, i:]
        ax.plot(tau, curve, lw=2, color=color, label=rf"$t_1 =${t1:g} s")
    # ax.text(0.04, 0.94, panel["label"], transform=ax.transAxes, ha="left", va="top")
    ax.set_xlabel(r"Lag time $\tau$ (s)")
    ax.set_xlim(0, 2500)
    ax.set_xticks([0, 1250, 2500])
    ax.set_ylim(0.98, 1.22)
axes[0].set_ylabel(r"$f(\tau;t_1)=g_2(t_1,t_1+\tau)$")
axes[-1].legend(frameon=True, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.savefig(OUTPUT_DIR / "simulation_noneq_regions_correlation_slices.pdf", bbox_inches="tight")
plt.show()

In [ ]:
simulation_residuals = [panel["g2"] - panel["diag_mean"] for panel in simulation_panels]
simulation_residual_limit = max(float(np.nanmax(np.abs(residual))) for residual in simulation_residuals)
simulation_residual_limit = min(max(simulation_residual_limit, 0.02), 0.2)

fig, axes = plt.subplots(1, 4, figsize=(30, 6), sharex=True, sharey=True)
image = None
norm = TwoSlopeNorm(vmin=-simulation_residual_limit, vcenter=0.0, vmax=simulation_residual_limit)
for ax, panel, residual in zip(axes, simulation_panels, simulation_residuals):
    image = ax.imshow(
        residual,
        origin="lower",
        extent=[panel["times"][0], panel["times"][-1], panel["times"][0], panel["times"][-1]],
        cmap="PiYG",
        norm=norm,
        interpolation="nearest",
        rasterized=True,
    )
    # ax.text(0.04, 0.94, panel["label"], transform=ax.transAxes, ha="left", va="top")
    ax.set_xlabel(r"Time $t_2$ (s)")
    ax.set_xticks([0, 1250, 2500])
    ax.set_yticks([0, 1250, 2500])
axes[0].set_ylabel(r"Time $t_1$ (s)")
colorbar = fig.colorbar(image, ax=axes, location="right", pad=0.035)
colorbar.set_label(r"$g_2(t_1,t_2)-g_2^{\mathrm{TTI}}(t_2-t_1)$")
fig.savefig(OUTPUT_DIR / "simulation_noneq_regions_g2_minus_diag_mean.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from scipy.interpolate import griddata
from matplotlib.lines import Line2D

phase_pairs = [
    ("D", "gamma", "log", "linear", r"Diffusivity $D$ (cm$^2$$\cdot$s$^{-1}$)", r"GB stiffness $\Gamma$ (s$\cdot$cm$^{-2}$)"),
    # ("gamma", "GB_conc", "linear", "linear", r"GB stiffness $\Gamma$ (s$\cdot$cm$^{-2}$)", r"Effective GB concentration $\lambda_{\mathrm{GB}}$"),
    ("D", "GB_conc", "log", "linear", r"Diffusivity $D$ (cm$^2$$\cdot$s$^{-1}$)", r"Effective GB concentration $\lambda_{\mathrm{GB}}$"),
]

marker_palette = ['#4dbbd5'] * 4
marker_size = 400
marker_styles = {
    1: {"marker": "o", "facecolor": marker_palette[0], "edgecolor": "black", "size": marker_size, "linewidth": 1.8},
    2: {"marker": "s", "facecolor": marker_palette[1], "edgecolor": "black", "size": marker_size, "linewidth": 1.8},
    3: {"marker": "^", "facecolor": marker_palette[2], "edgecolor": "black", "size": marker_size, "linewidth": 1.8},
    4: {"marker": "D", "facecolor": marker_palette[3], "edgecolor": "black", "size": marker_size, "linewidth": 1.8},
}

# Layout controls for the phase-diagram row.
phase_figsize = (23, 8)
phase_panel_gap = 0.35      # gap between phase-diagram panels, in units of one panel width
phase_colorbar_gap = 0.06   # gap between third panel and colorbar
phase_colorbar_width = 0.045

fig = plt.figure(figsize=phase_figsize)
grid = fig.add_gridspec(
    1,
    5,
    width_ratios=[
        1,
        phase_panel_gap,
        1,
        phase_colorbar_gap,
        phase_colorbar_width,
    ],
    wspace=0.0,
)
axes = [fig.add_subplot(grid[0, i]) for i in [0, 2]]
colorbar_ax = fig.add_subplot(grid[0, -1])

levels = np.linspace(0.0, 1.0, 101)
contour = None

def _set_tick_helper(label):
    if label.startswith("D"):
        return [1e-23, 1e-22, 1e-21]
    elif label.startswith("GB stiffness"):
        return [1e18, 2e18, 3e18, 4e18, 5e18, 6e18]
    elif label.startswith("Effective GB concentration"):
        return [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    else:
        raise ValueError(f"Unexpected label: {label}")
    

for ax, (x, y, xscale, yscale, xlabel, ylabel) in zip(axes, phase_pairs):
    X = simulation_df[x].to_numpy(dtype=np.float64)
    Y = simulation_df[y].to_numpy(dtype=np.float64)
    Z = simulation_df["nonequilibrium_measure_raw"].to_numpy(dtype=np.float64)

    x_transformed = np.log10(X) if xscale == "log" else X
    y_transformed = np.log10(Y) if yscale == "log" else Y

    grid_x_transformed = np.linspace(x_transformed.min(), x_transformed.max(), 400)
    grid_y_transformed = np.linspace(y_transformed.min(), y_transformed.max(), 400)
    grid_x_t, grid_y_t = np.meshgrid(grid_x_transformed, grid_y_transformed)

    grid_z_linear = griddata(
        points=np.column_stack([x_transformed, y_transformed]),
        values=Z,
        xi=(grid_x_t, grid_y_t),
        method="linear",
        rescale=True,
    )
    grid_z_nearest = griddata(
        points=np.column_stack([x_transformed, y_transformed]),
        values=Z,
        xi=(grid_x_t, grid_y_t),
        method="nearest",
        rescale=True,
    )
    grid_z = np.where(np.isfinite(grid_z_linear), grid_z_linear, grid_z_nearest)
    grid_z = np.clip(grid_z, 0.0, 1.0)

    grid_x = np.power(10.0, grid_x_t) if xscale == "log" else grid_x_t
    grid_y = np.power(10.0, grid_y_t) if yscale == "log" else grid_y_t

    contour = ax.contourf(
        grid_x,
        grid_y,
        grid_z,
        levels=levels,
        cmap="plasma",
        vmin=0.0,
        vmax=1.0,
    )
    for collection in getattr(contour, "collections", []):
        collection.set_rasterized(True)

    for _, row in selected_simulations.iterrows():
        region = int(row["noneq_region"])
        style = marker_styles[region]
        ax.scatter(
            row[x],
            row[y],
            s=style["size"],
            marker=style["marker"],
            facecolors=style["facecolor"],
            edgecolors=style["edgecolor"],
            linewidths=style["linewidth"],
            zorder=5,
        )

    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(_set_tick_helper(xlabel))
    ax.set_yticks(_set_tick_helper(ylabel))

legend_handles = []
for i, (_, row) in enumerate(selected_simulations.sort_values("noneq_region").iterrows()):
    region = int(row["noneq_region"])
    style = marker_styles[region]
    legend_handles.append(
        Line2D(
            [0], [0],
            marker=style["marker"],
            linestyle="None",
            markerfacecolor=style["facecolor"],
            markeredgecolor=style["edgecolor"],
            markeredgewidth=style["linewidth"],
            markersize=14,
            label=f'Q{i+1}',
        )
    )
ax.legend(handles=legend_handles, frameon=True, loc="upper left", bbox_to_anchor=(1.35, 1.0))

colorbar = fig.colorbar(contour, cax=colorbar_ax)
colorbar.set_label(r"Non-equilibrium measure")
colorbar.set_ticks([0.0, 0.5, 1.0])
fig.savefig(OUTPUT_DIR / "simulation_noneq_regions_phase_diagrams.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# --- ML prediction section ---
# Simulated pred-vs-true figures for the best vanilla no-T model and the CORAL-surrogate no-T model.
from torch.utils.data import DataLoader
from scipy.interpolate import griddata
import os

# The training dataset classes load tensor paths exactly as stored in manifest.csv,
# so keep the kernel cwd at the repository root before constructing those datasets.
os.chdir(PROJECT_ROOT)

ML_SIM_ROOT = PROJECT_ROOT / "dataset" / "simulation"
ML_BATCH_SIZE = 64
ML_NUM_WORKERS = 0
ML_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VANILLA_MODEL_PATH = PROJECT_ROOT / "models" / "Vanilla_XPCS_no_T_best_20260414-202028.pt"
CORAL_SURROGATE_MODEL_PATH = PROJECT_ROOT / "models" / "XPCS_coral_surrogate_no_T_best_20260421-000521.pt"

ML_OUTPUT_STEM = {
    "vanilla": "ml_vanilla",
    "coral_surrogate": "ml_coral_surrogate",
}


In [ ]:
# --- ML prediction section ---
def _phase_noneq_table(sim_root: Path) -> pd.DataFrame:
    manifest = pd.read_csv(sim_root / "manifest.csv")
    noneq_col = "nonequilibrium_measure_raw" if "nonequilibrium_measure_raw" in manifest.columns else "nonequilibrium_measure"
    required = ["D", "gamma", "GB_conc", noneq_col]
    missing = [col for col in required if col not in manifest.columns]
    if missing:
        raise ValueError(f"Missing columns in {sim_root / 'manifest.csv'}: {missing}")
    table = manifest[required].rename(columns={noneq_col: "S_noneq_raw"}).dropna().copy()
    table = table[table["D"] > 0]
    return table


def _interpolate_raw_noneq(params: pd.DataFrame, phase_table: pd.DataFrame) -> np.ndarray:
    points = np.column_stack([
        np.log10(phase_table["D"].to_numpy(dtype=float)),
        phase_table["gamma"].to_numpy(dtype=float),
        phase_table["GB_conc"].to_numpy(dtype=float),
    ])
    values = phase_table["S_noneq_raw"].to_numpy(dtype=float)
    d_values = np.clip(params["D"].to_numpy(dtype=float), phase_table["D"].min(), phase_table["D"].max())
    xi = np.column_stack([
        np.log10(d_values),
        params["gamma"].to_numpy(dtype=float),
        params["GB_conc"].to_numpy(dtype=float),
    ])
    linear = griddata(points, values, xi, method="linear", rescale=True)
    nearest = griddata(points, values, xi, method="nearest", rescale=True)
    return np.where(np.isfinite(linear), linear, nearest)


def _predict_simulated_dataset(model_label, model_path, dataset_cls, load_model_fn, denorm_fn):
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(model_path)

    dataset = dataset_cls(ML_SIM_ROOT)
    loader = DataLoader(dataset, batch_size=ML_BATCH_SIZE, shuffle=False, num_workers=ML_NUM_WORKERS)
    model = load_model_fn(model_path, ML_DEVICE)
    model.eval()

    pred_raw_batches = []
    true_raw_batches = []
    with torch.no_grad():
        for batch in loader:
            x, y_norm, y_raw, T = batch[:4]
            x = x.to(ML_DEVICE)
            T = T.to(ML_DEVICE)
            pred_norm = model(x, T)
            pred_raw = denorm_fn(pred_norm, model.norm_meta, device=ML_DEVICE)
            pred_raw_batches.append(pred_raw.detach().cpu().numpy())
            true_raw_batches.append(y_raw.detach().cpu().numpy())

    pred_raw = np.concatenate(pred_raw_batches, axis=0)
    true_raw = np.concatenate(true_raw_batches, axis=0)
    manifest = dataset.manifest.reset_index(drop=True).copy()

    pred_df = pd.DataFrame({
        "id": manifest["id"].to_numpy() if "id" in manifest.columns else np.arange(len(manifest)),
        "T": manifest["T"].to_numpy(dtype=float),
        "gamma_true": true_raw[:, 0],
        "D_true": true_raw[:, 1],
        "GB_conc_true": true_raw[:, 2],
        "gamma_pred": pred_raw[:, 0],
        "D_pred": pred_raw[:, 1],
        "GB_conc_pred": pred_raw[:, 2],
    })

    phase_table = _phase_noneq_table(ML_SIM_ROOT)
    true_params = pred_df.rename(columns={"gamma_true": "gamma", "D_true": "D", "GB_conc_true": "GB_conc"})
    pred_params = pred_df.rename(columns={"gamma_pred": "gamma", "D_pred": "D", "GB_conc_pred": "GB_conc"})
    if "nonequilibrium_measure_raw" in manifest.columns:
        pred_df["S_noneq_true"] = manifest["nonequilibrium_measure_raw"].to_numpy(dtype=float)
    else:
        pred_df["S_noneq_true"] = _interpolate_raw_noneq(true_params[["D", "gamma", "GB_conc"]], phase_table)
    pred_df["S_noneq_pred"] = _interpolate_raw_noneq(pred_params[["D", "gamma", "GB_conc"]], phase_table)

    csv_path = OUTPUT_DIR / f"{ML_OUTPUT_STEM[model_label]}_predictions.csv"
    pred_df.to_csv(csv_path, index=False)
    print(f"{model_label}: wrote {csv_path.relative_to(PROJECT_ROOT)}")
    return pred_df


In [ ]:
# --- ML prediction section ---
from train_vanilla_no_T import (
    XPCSDataset as VanillaXPCSDataset,
    denorm_from_meta as denorm_vanilla_params,
    load_model as load_vanilla_model,
)
from train_adv_coral_surrogate import (
    XPCSDataset as CoralSurrogateXPCSDataset,
    denorm_from_meta as denorm_coral_params,
    load_model as load_coral_surrogate_model,
)

ml_prediction_frames = {}
ml_prediction_frames["vanilla"] = _predict_simulated_dataset(
    "vanilla",
    VANILLA_MODEL_PATH,
    VanillaXPCSDataset,
    load_vanilla_model,
    denorm_vanilla_params,
)

ml_prediction_frames["coral_surrogate"] = _predict_simulated_dataset(
    "coral_surrogate",
    CORAL_SURROGATE_MODEL_PATH,
    CoralSurrogateXPCSDataset,
    load_coral_surrogate_model,
    denorm_coral_params,
)


In [ ]:
# --- ML prediction section ---
# Pred-vs-true figures: one standalone S_noneq figure and one 3-panel parameter figure per model.
ml_model_styles = {
    "vanilla": {"color": "#2f6fbb", "marker": "o", "label": "Vanilla"},
    "coral_surrogate": {"color": "#b84a62", "marker": "s", "label": "CORAL-surrogate"},
}

ml_param_specs = [
    {
        "name": "D",
        "true": "D_true",
        "pred": "D_pred",
        "scale": 1e22,
        "xlabel": r"Real $D$ ($10^{-22}$ cm$^2\cdot$s$^{-1}$)",
        "ylabel": r"Predicted $D$ ($10^{-22}$ cm$^2\cdot$s$^{-1}$)",
    },
    # {
    #     "name": "gamma",
    #     "true": "gamma_true",
    #     "pred": "gamma_pred",
    #     "scale": 1e-18,
    #     "xlabel": r"Real $\Gamma$ ($10^{18}$ s$\cdot$cm$^{-2}$)",
    #     "ylabel": r"Predicted $\Gamma$ ($10^{18}$ s$\cdot$cm$^{-2}$)",
    # },
    {
        "name": "GB_conc",
        "true": "GB_conc_true",
        "pred": "GB_conc_pred",
        "scale": 1.0,
        "xlabel": r"Real $\lambda_{\mathrm{GB}}$",
        "ylabel": r"Predicted $\lambda_{\mathrm{GB}}$",
    },
]

# Plain-text R^2 values for the quantities plotted below.
def _r2_score_np(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if x.size < 2:
        return np.nan
    ss_res = np.sum((y - x) ** 2)
    ss_tot = np.sum((x - x.mean()) ** 2)
    if ss_tot == 0:
        return np.nan
    return 1.0 - ss_res / ss_tot

ml_r2_specs = [
    {"label": "S_noneq", "true": "S_noneq_true", "pred": "S_noneq_pred", "scale": 1.0},
    *[{"label": spec["name"], "true": spec["true"], "pred": spec["pred"], "scale": spec["scale"]} for spec in ml_param_specs],
]

for model_label, pred_df in ml_prediction_frames.items():
    if pred_df is None:
        continue
    print(ml_model_styles[model_label]["label"])
    for spec in ml_r2_specs:
        x = pred_df[spec["true"]].to_numpy(dtype=float) * spec["scale"]
        y = pred_df[spec["pred"]].to_numpy(dtype=float) * spec["scale"]
        print(f"  {spec['label']}: R^2 = {_r2_score_np(x, y):.4f}")

for model_label, pred_df in ml_prediction_frames.items():
    if pred_df is None:
        continue
    style = ml_model_styles[model_label]
    stem = ML_OUTPUT_STEM[model_label]

    fig, ax = plt.subplots(figsize=(3.2, 3))
    x = pred_df["S_noneq_true"].to_numpy(dtype=float)
    y = pred_df["S_noneq_pred"].to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    lo = min(float(x.min()), float(y.min()))
    hi = max(float(x.max()), float(y.max()))
    pad = 0.04 * (hi - lo if hi > lo else 1.0)
    ax.scatter(x, y, s=18, marker=style["marker"], color=style["color"], alpha=0.42, edgecolors="none")
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="0.20", lw=1.1, ls="--")
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel(r"Real $\mathcal{S}_{\mathrm{noneq}}$")
    ax.set_ylabel(r"Predicted $\mathcal{S}_{\mathrm{noneq}}$")
    fig.savefig(OUTPUT_DIR / f"{stem}_pred_vs_true_noneq.pdf", bbox_inches="tight")
    plt.show()

    fig = plt.figure(figsize=(17, 7))
    grid = fig.add_gridspec(
        1,
        3,
        width_ratios=[
            1,
            0.06,
            1,
        ]
    )
    axes = [fig.add_subplot(grid[0, i]) for i in [0, 2]]
    for ax, spec in zip(axes, ml_param_specs):
        x = pred_df[spec["true"]].to_numpy(dtype=float) * spec["scale"]
        y = pred_df[spec["pred"]].to_numpy(dtype=float) * spec["scale"]
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        lo = min(float(x.min()), float(y.min()))
        hi = max(float(x.max()), float(y.max()))
        pad = 0.04 * (hi - lo if hi > lo else 1.0)
        ax.scatter(x, y, s=15, marker=style["marker"], color=style["color"], alpha=0.42, edgecolors="none")
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="0.20", lw=1.0, ls="--")
        ax.set_xlim(lo - pad, hi + pad)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xlabel(spec["xlabel"])
        ax.set_ylabel(spec["ylabel"])
        if spec["name"] == "D":
            ax.set_xticks([0, 5, 10])
            ax.set_yticks([0, 5, 10])
        ax.text(
            0.04, 0.95,
            rf"$R^2_\mathrm{{Vanilla}}$ = {_r2_score_np(x, y):.3f}" if model_label == "vanilla" else rf"$R^2_\mathrm{{Adapted}}$ = {_r2_score_np(x, y):.3f}",
            transform=ax.transAxes,
            ha='left', va='top', fontsize=24,
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#d1d5db', lw=1.0, alpha=0.96)
        )
    fig.savefig(OUTPUT_DIR / f"{stem}_pred_vs_true_parameters.pdf", bbox_inches="tight")
    plt.show()
    plt.close('all')


In [ ]:
# --- Experiment phase diagram and UMAP section ---
# Experiment phase diagrams, laid out explicitly from saved result CSVs.
# Background uses the broader simulation_2 parameter domain.
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from scipy.interpolate import griddata
from inference import load_overlay_points, add_overlay_metadata

EXP_PHASE_SIM_MANIFEST = PROJECT_ROOT / "dataset" / "simulation_2" / "manifest.csv"
EXP_PHASE_MODEL_SPECS = [
    {
        "label": "Vanilla",
        "model_name": "vanilla",
        "results_dir": PROJECT_ROOT / "results" / "vanilla_noT_vanseed42_top-left_20260414-202239",
        "output": "experiment_phase_diagrams_vanilla.pdf",
    },
    {
        "label": "Contrastive",
        "model_name": "adv",
        "results_dir": PROJECT_ROOT / "results" / "coral-surrogate-material-ft_noT_shot0_predictor_mse_seed42_all-diagonal_20260424-181439",
        "output": "experiment_phase_diagrams_contrastive.pdf",
    },
]

exp_phase_df = pd.read_csv(EXP_PHASE_SIM_MANIFEST)
exp_phase_value_column = "nonequilibrium_measure_raw"
exp_overlay_value_column = "experimental_nonequilibrium_measure"

exp_phase_specs = [
    # {
    #     "x": "D",
    #     "y": "gamma",
    #     "xscale": "log",
    #     "yscale": "linear",
    #     "xlabel": r"Diffusivity $D$ (cm$^2\cdot$s$^{-1}$)",
    #     "ylabel": r"GB stiffness $\Gamma$ ($10^{18}$ s$\cdot$cm$^{-2}$)",
    #     "xticks": [1e-23, 1e-22, 1e-21],
    #     "xlabels": [r"$10^{-23}$", r"$10^{-22}$", r"$10^{-21}$"],
    #     "yticks": [1e18, 2e18, 3e18, 4e18, 5e18, 6e18],
    #     "ylabels": ["1", "2", "3", "4", "5", "6"],
    #     "x_range": (3e-24, 3e-21),
    #     "y_range": (1e18, 6e18),
    # },
    # {
    #     "x": "gamma",
    #     "y": "GB_conc",
    #     "xscale": "linear",
    #     "yscale": "linear",
    #     "xlabel": r"GB stiffness $\Gamma$ ($10^{18}$ s$\cdot$cm$^{-2}$)",
    #     "ylabel": r"Effective GB concentration $\lambda_{\mathrm{GB}}$",
    #     "xticks": [1e18, 2e18, 3e18, 4e18, 5e18, 6e18],
    #     "xlabels": ["1", "2", "3", "4", "5", "6"],
    #     "yticks": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    #     "ylabels": ["0", "0.1", "0.2", "0.3", "0.4", "0.5"],
    #     "x_range": (1e18, 6e18),
    #     "y_range": (0.0, 0.5),
    # },
    {
        "x": "D",
        "y": "GB_conc",
        "xscale": "log",
        "yscale": "linear",
        "xlabel": r"Diffusivity $D$ (cm$^2\cdot$s$^{-1}$)",
        "ylabel": r"Effective GB concentration $\lambda_{\mathrm{GB}}$",
        "xticks": [1e-23, 1e-22, 1e-21],
        "xlabels": [r"$10^{-23}$", r"$10^{-22}$", r"$10^{-21}$"],
        "yticks": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
        "ylabels": ["0", "0.1", "0.2", "0.3", "0.4", "0.5"],
        "x_range": (3e-24, 3e-21),
        "y_range": (0.0, 0.5),
    },
]

exp_phase_figsize = (8, 7.0)
exp_phase_panel_gap = 0.16
exp_phase_colorbar_gap = 0.05
exp_phase_colorbar_width = 0.045
exp_phase_grid_resolution = 400
exp_phase_marker_size = 450
exp_phase_marker_linewidth = 2.0
exp_phase_show_temperature_legend = False
exp_phase_cmap = "plasma"
exp_phase_norm = Normalize(vmin=0.0, vmax=1.0)

for model_spec in EXP_PHASE_MODEL_SPECS:
    exp_overlay_df = load_overlay_points(
        results_dir=model_spec["results_dir"],
        model_name=model_spec["model_name"],
        aggregate_by="shot",
        shot_index=0,
    )
    exp_overlay_df = add_overlay_metadata(exp_overlay_df, exp_phase_df)

    fig = plt.figure(figsize=exp_phase_figsize)
    grid = fig.add_gridspec(
        1,
        2 * len(exp_phase_specs) + 1,
        width_ratios=[
            1,
            exp_phase_colorbar_gap,
            exp_phase_colorbar_width,
        ],
    )
    axes = [fig.add_subplot(grid[0, i]) for i in [0]]
    cax = fig.add_subplot(grid[0, -2])
    last_contour = None

    for ax, spec in zip(axes, exp_phase_specs):
        x_values = exp_phase_df[spec["x"]].to_numpy(dtype=float)
        y_values = exp_phase_df[spec["y"]].to_numpy(dtype=float)
        z_values = exp_phase_df[exp_phase_value_column].to_numpy(dtype=float)
        x_transformed = np.log10(x_values) if spec["xscale"] == "log" else x_values
        y_transformed = np.log10(y_values) if spec["yscale"] == "log" else y_values
        xt_lo = np.log10(spec["x_range"][0]) if spec["xscale"] == "log" else spec["x_range"][0]
        xt_hi = np.log10(spec["x_range"][1]) if spec["xscale"] == "log" else spec["x_range"][1]
        yt_lo = np.log10(spec["y_range"][0]) if spec["yscale"] == "log" else spec["y_range"][0]
        yt_hi = np.log10(spec["y_range"][1]) if spec["yscale"] == "log" else spec["y_range"][1]

        grid_x_t, grid_y_t = np.meshgrid(
            np.linspace(xt_lo, xt_hi, exp_phase_grid_resolution),
            np.linspace(yt_lo, yt_hi, exp_phase_grid_resolution),
        )
        grid_z = griddata(
            np.column_stack([x_transformed, y_transformed]),
            z_values,
            (grid_x_t, grid_y_t),
            method="linear",
            rescale=True,
        )
        nearest_grid_z = griddata(
            np.column_stack([x_transformed, y_transformed]),
            z_values,
            (grid_x_t, grid_y_t),
            method="nearest",
            rescale=True,
        )
        grid_z = np.where(np.isfinite(grid_z), grid_z, nearest_grid_z)
        grid_x = np.power(10.0, grid_x_t) if spec["xscale"] == "log" else grid_x_t
        grid_y = np.power(10.0, grid_y_t) if spec["yscale"] == "log" else grid_y_t

        last_contour = ax.contourf(
            grid_x,
            grid_y,
            grid_z,
            levels=np.linspace(0.0, 1.0, 101),
            cmap=exp_phase_cmap,
            norm=exp_phase_norm,
        )

        ax.set_xscale(spec["xscale"])
        ax.set_yscale(spec["yscale"])
        ax.set_xlim(*spec["x_range"])
        ax.set_ylim(*spec["y_range"])
        ax.set_xlabel(spec["xlabel"])
        ax.set_ylabel(spec["ylabel"])
        ax.set_xticks(spec["xticks"], spec["xlabels"])
        ax.set_yticks(spec["yticks"], spec["ylabels"])

        overlay_x = exp_overlay_df[spec["x"]].to_numpy(dtype=float)
        overlay_y = exp_overlay_df[spec["y"]].to_numpy(dtype=float)
        overlay_value = exp_overlay_df[exp_overlay_value_column].to_numpy(dtype=float)
        visible = (
            np.isfinite(overlay_x)
            & np.isfinite(overlay_y)
            & np.isfinite(overlay_value)
            & (overlay_x >= spec["x_range"][0])
            & (overlay_x <= spec["x_range"][1])
            & (overlay_y >= spec["y_range"][0])
            & (overlay_y <= spec["y_range"][1])
        )
        visible_df = exp_overlay_df.loc[visible].copy()

        for _, style in (
            visible_df[
                ["temperature_c", "temperature_order", "temperature_label", "temperature_color", "temperature_marker"]
            ]
            .drop_duplicates()
            .sort_values(["temperature_order", "temperature_c"])
            .iterrows()
        ):
            group = visible_df.loc[visible_df["temperature_c"] == style["temperature_c"]]
            ax.scatter(
                group[spec["x"]],
                group[spec["y"]],
                c=group[exp_overlay_value_column],
                cmap=exp_phase_cmap,
                norm=exp_phase_norm,
                s=exp_phase_marker_size,
                marker=str(style["temperature_marker"]),
                edgecolors=str(style["temperature_color"]),
                linewidths=exp_phase_marker_linewidth,
                alpha=0.95,
                zorder=4,
            )

    if exp_phase_show_temperature_legend:
        temperature_handles = []
        for _, style in (
            exp_overlay_df[
                ["temperature_c", "temperature_order", "temperature_label", "temperature_color", "temperature_marker"]
            ]
            .drop_duplicates()
            .sort_values(["temperature_order", "temperature_c"])
            .iterrows()
        ):
            temperature_handles.append(
                Line2D(
                    [0],
                    [0],
                    marker=str(style["temperature_marker"]),
                    color="none",
                    markerfacecolor="white",
                    markeredgecolor=str(style["temperature_color"]),
                    markersize=9,
                    linewidth=0,
                    label=str(style["temperature_label"]),
                )
            )
        axes[-1].legend(
            handles=temperature_handles,
            loc="center left",
            bbox_to_anchor=(1.08, 0.5),
            fontsize=9,
            frameon=False,
            ncol=2,
        )

    cbar = fig.colorbar(last_contour, cax=cax)
    cbar.set_label(r"Non-equilibrium measure")
    cbar.set_ticks([0.0, 0.5, 1.0])
    display(fig)
    fig.savefig(OUTPUT_DIR / model_spec["output"], bbox_inches="tight")
    plt.close(fig)


In [ ]:
# --- Experiment phase diagram and UMAP section ---
# UMAP for vanilla and contrastive models. Data preparation follows the original run_all.py evaluation path.
from IPython.display import display
import umap.umap_ as umap
from torch.utils.data import DataLoader, Subset
from run_all import PreparedExperimentDataset, prepare_experiment_shots
from train_adv_no_T import XPCSDataset as EvalUMAPSimulationDataset
from train_vanilla_no_T import load_model as load_vanilla_umap_model
from train_adv_coral_distill import load_model as load_contrastive_umap_model

UMAP_SIM_ROOT = PROJECT_ROOT / "dataset" / "simulation"
UMAP_RAW_DATA_DIR = PROJECT_ROOT / "exp_data"
UMAP_VANILLA_MODEL_PATH = PROJECT_ROOT / "models" / "Vanilla_XPCS_no_T_best_20260414-202028.pt"
UMAP_CONTRASTIVE_MODEL_PATH = PROJECT_ROOT / "models" / "XPCS_coral_no_T_best_20260422-003632.pt"
UMAP_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
UMAP_SIM_LIMIT = None
UMAP_BATCH_SIZE = 32
UMAP_NUM_WORKERS = 2
UMAP_N_NEIGHBORS = 5
UMAP_MIN_DIST = 0.1
UMAP_INIT = "spectral"
UMAP_RANDOM_STATE = 42
UMAP_CROP_SIZE = 2500
UMAP_COARSE_SIZE = 256
UMAP_CROP_STEP = 100
UMAP_CROP_POLICY = "top-left"
UMAP_NO_T = True


def _maybe_limit_umap_dataset(dataset, limit):
    if limit is None or limit <= 0 or limit >= len(dataset):
        return dataset
    indices = np.linspace(0, len(dataset) - 1, num=limit, dtype=int).tolist()
    return Subset(dataset, indices)


def _raw_files_from_result_dir(result_dir):
    file_names = []
    for csv_path in sorted(Path(result_dir).glob("*/*.csv")):
        if csv_path.parent.name.startswith("phase_diagrams"):
            continue
        frame = pd.read_csv(csv_path, usecols=["file_name"])
        if not frame.empty:
            file_names.append(str(frame["file_name"].iloc[0]))
    return [UMAP_RAW_DATA_DIR / name for name in sorted(set(file_names))]


def _extract_umap_features(model, dataset, device):
    loader = DataLoader(dataset, batch_size=UMAP_BATCH_SIZE, shuffle=False, num_workers=UMAP_NUM_WORKERS)
    model.eval()
    model.to(device)
    features = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(device)
            T = batch[3].to(device)
            if hasattr(model, "extract_features") and hasattr(model, "build_shared_features"):
                xpcs_features, temp_features = model.extract_features(x, T)
                latent = model.build_shared_features(xpcs_features, temp_features)
            else:
                latent = model.conv_net(x)
            features.append(latent.detach().cpu())
    return torch.cat(features, dim=0).numpy()


umap_model_specs = [
    {
        "label": "Vanilla",
        "result_dir": PROJECT_ROOT / "results" / "vanilla_noT_vanseed42_top-left_20260414-202239",
        "load_model": load_vanilla_umap_model,
        "model_path": UMAP_VANILLA_MODEL_PATH,
        "sim_marker": "s",
        "exp_marker": "^",
        "sim_marker_size": 400,
        "exp_marker_size": 400,
        "output": "UMAP_vanilla.pdf",
    },
    {
        "label": "Contrastive",
        "result_dir": PROJECT_ROOT / "results" / "coral_noT_contrastive-soft-infonce_cw1_top-left_20260422-004405",
        "load_model": load_contrastive_umap_model,
        "model_path": UMAP_CONTRASTIVE_MODEL_PATH,
        "sim_marker": "o",
        "exp_marker": "o",
        "sim_marker_size": 400,
        "exp_marker_size": 400,
        "output": "UMAP_contrastive.pdf",
    },
]

for spec in umap_model_specs:
    sim_dataset = _maybe_limit_umap_dataset(EvalUMAPSimulationDataset(UMAP_SIM_ROOT), UMAP_SIM_LIMIT)
    raw_files = _raw_files_from_result_dir(spec["result_dir"])
    samples = prepare_experiment_shots(
        raw_files=raw_files,
        shot_indices=None,
        crop_size=UMAP_CROP_SIZE,
        coarse_size=UMAP_COARSE_SIZE,
        crop_step=UMAP_CROP_STEP,
        crop_policy=UMAP_CROP_POLICY,
        no_t=UMAP_NO_T,
        cache_dir=PROJECT_ROOT / "dataset" / "experiment_eval_cache",
    )
    exp_dataset = PreparedExperimentDataset(samples)
    model = spec["load_model"](spec["model_path"], UMAP_DEVICE)

    sim_features = _extract_umap_features(model, sim_dataset, UMAP_DEVICE)
    exp_features = _extract_umap_features(model, exp_dataset, UMAP_DEVICE)
    all_features = np.vstack([sim_features, exp_features])
    domain_labels = np.concatenate([
        np.zeros(sim_features.shape[0], dtype=int),
        np.ones(exp_features.shape[0], dtype=int),
    ])
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=max(2, min(UMAP_N_NEIGHBORS, all_features.shape[0] - 1)),
        min_dist=UMAP_MIN_DIST,
        metric="euclidean",
        init=UMAP_INIT,
        random_state=UMAP_RANDOM_STATE,
    )
    coords = reducer.fit_transform(all_features)

    fig, ax = plt.subplots(figsize=(8, 6))
    sim_mask = domain_labels == 0
    exp_mask = domain_labels == 1
    ax.scatter(
        coords[sim_mask, 0],
        coords[sim_mask, 1],
        s=spec["sim_marker_size"],
        marker=spec["sim_marker"],
        color="#4DBBD5FF",
        alpha=1.0,
        edgecolors="none",
        label="Simulation",
    )
    ax.scatter(
        coords[exp_mask, 0],
        coords[exp_mask, 1],
        s=spec["exp_marker_size"],
        marker=spec["exp_marker"],
        color="#F39B7FFF",
        alpha=1.0,
        edgecolors="none",
        label="Experiment",
    )
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
    ax.set_xlim(x_min - 0.10 * (x_max - x_min), x_max + 0.10 * (x_max - x_min))
    ax.set_ylim(y_min - 0.10 * (y_max - y_min), y_max + 0.10 * (y_max - y_min))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    display(fig)
    fig.savefig(OUTPUT_DIR / spec["output"], bbox_inches="tight")
    plt.close(fig)
